# Joint Sharp Wave Ripple Detection

This tutorial walks through a compact joint SWR workflow in `neuro_py` using the session `S:\data\HMC\HMC1\day8`.

We will:

1. load the session and infer the ripple, sharp-wave, and optional noise channels from CellExplorer tags,
2. run the joint detector so that accepted events must contain both a ripple and a sharp wave,
3. inspect the CellExplorer-style event table, and
4. visualize the raw LFP together with the ripple and sharp-wave features that drove the detection.


In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from scipy import ndimage, signal

import neuro_py as npy
from neuro_py.detectors.sharp_wave_ripple import detect_sharp_wave_ripples

sns.set_theme(style="whitegrid", context="talk")
npy.plotting.set_plotting_defaults()


## 1) Detect joint SWRs from the session

The detector will look for CellExplorer `channelTags` so that it can automatically find:

- the ripple channel from tags such as `ripple` or `CA1sp`,
- the sharp-wave channel from tags such as `SharpWave` or `CA1sr`, and
- an optional noise channel from the `Bad` tag.

The key point is that this is now a **joint** detector: ripple-band power must cross its own thresholds **and** a nearby sharp-wave event must cross separate sharp-wave thresholds. In this example we also ask the detector to use the sharp-wave interval as the final event boundary.


In [ ]:
basepath = r"S:\data\HMC\HMC1\day8"

swr_params = dict(
    low_threshold=0.75,
    high_threshold=2.5,
    sharp_wave_low_threshold=0.25,
    sharp_wave_high_threshold=1.5,
    min_duration=0.020,
    max_duration=0.100,
    sharp_wave_min_duration=0.020,
    sharp_wave_max_duration=0.250,
    noise_threshold=5.0,
    merge_gap=0.010,
    boundary_mode="sharp_wave",
    save_mat=True,
    # event_name="ripples_test",
    # overwrite=True,
)

ripples = detect_sharp_wave_ripples(basepath=basepath, **swr_params)
ripples.head()


## 2) Inspect the joint detection output

The returned table still follows the CellExplorer event convention, but now it also exposes fields that are useful for joint SWR QC. In particular:

- `peakNormedPower` reports the ripple feature strength,
- `sharp_wave_peakNormedPower` reports the sharp-wave feature strength, and
- `sharp_wave_duration` lets us confirm that the accepted event had a plausible low-frequency component.


In [ ]:
summary_columns = [
    "start",
    "stop",
    "peaks",
    "duration",
    "ripple_duration",
    "sharp_wave_duration",
    "frequency",
    "peakNormedPower",
    "sharp_wave_peakNormedPower",
    "noise_peakNormedPower",
]

print(f"Detected {len(ripples):,} joint SWR events")
display(ripples[summary_columns].head(10))


## 3) Recreate the features used by the detector

For visualization it helps to rebuild the two signals that drive the joint decision:

- a ripple-band feature from the ripple channel, and
- a low-frequency sharp-wave difference signal between the ripple and sharp-wave channels.

That makes it easy to see why a given interval was accepted as an SWR instead of being just a ripple-like oscillation or a broad low-frequency deflection.


In [ ]:
channel_tags = npy.io.load_channel_tags(basepath)

def first_tagged_channel(tags, *keys):
    for key in keys:
        if key in tags:
            return int(np.atleast_1d(tags[key]["channels"])[0] - 1)
    return None

def zscore(values):
    values = np.asarray(values, dtype=float)
    return (values - np.nanmean(values)) / np.nanstd(values)

ripple_channel = (
    int(ripples["ripple_channel"].iloc[0])
    if (not ripples.empty and "ripple_channel" in ripples.columns and not np.isnan(ripples["ripple_channel"].iloc[0]))
    else first_tagged_channel(channel_tags, "ripple", "Ripple", "CA1sp", "ca1sp")
)
sharp_wave_channel = first_tagged_channel(channel_tags, "SharpWave", "sharpwave", "sharp_wave", "CA1sr", "ca1sr")
noise_channel = first_tagged_channel(channel_tags, "Bad", "bad")

channels = [ripple_channel]
if sharp_wave_channel is not None:
    channels.append(sharp_wave_channel)
if noise_channel is not None and noise_channel not in channels:
    channels.append(noise_channel)

lfp = npy.io.LFPLoader(basepath, channels=channels, ext="lfp")
fs = float(lfp.fs)
timestamps = np.asarray(lfp.abscissa_vals)
lfp_data = np.asarray(lfp.data)

channel_lookup = {channel: idx for idx, channel in enumerate(channels)}
raw = np.asarray(lfp_data[channel_lookup[ripple_channel]]).squeeze()
sharp_wave_raw = None
if sharp_wave_channel is not None:
    sharp_wave_raw = np.asarray(lfp_data[channel_lookup[sharp_wave_channel]]).squeeze()

ripple_sos = signal.butter(4, (120.0, 250.0), btype="bandpass", fs=fs, output="sos")
ripple_filtered = signal.sosfiltfilt(ripple_sos, raw)
ripple_envelope = np.abs(signal.hilbert(ripple_filtered))
ripple_envelope = ndimage.gaussian_filter1d(ripple_envelope, sigma=swr_params.get("smooth_sigma", 0.004) * fs, mode="nearest")
ripple_power = zscore(ripple_envelope)

sharp_wave_diff = None
sharp_wave_power = None
if sharp_wave_raw is not None:
    sharp_sos = signal.butter(4, (2.0, 50.0), btype="bandpass", fs=fs, output="sos")
    ripple_low = signal.sosfiltfilt(sharp_sos, raw)
    sharp_low = signal.sosfiltfilt(sharp_sos, sharp_wave_raw)
    sharp_wave_diff = ripple_low - sharp_low
    sharp_wave_power = zscore(sharp_wave_diff)

print(f"Ripple channel: {ripple_channel}")
print(f"Sharp-wave channel: {sharp_wave_channel}")
print(f"Noise channel: {noise_channel}")
print(f"Sampling rate: {fs:.1f} Hz")


## 4) Plot the joint detector features

The overview plot shows the raw ripple channel together with a scaled sharp-wave difference trace so the accepted intervals stand out against the ongoing LFP.

Each example panel then shows:

- the raw ripple channel,
- the ripple-band filtered signal,
- the sharp-wave difference trace, and
- the z-scored ripple and sharp-wave features with their respective high thresholds.

That gives a quick visual check that every accepted event really satisfies the **joint** criterion.


In [ ]:
def plot_joint_swr_summary(
    events,
    timestamps,
    raw_signal,
    ripple_filtered,
    ripple_power,
    fs,
    params,
    sharp_wave_diff=None,
    sharp_wave_power=None,
    overview_seconds=20.0,
    example_window=0.10,
    n_examples=6,
):
    fig = plt.figure(
        figsize=npy.plotting.set_size("nature_double", fraction=1.6),
        constrained_layout=True,
        dpi=200,
    )
    grid = fig.add_gridspec(2, 1, height_ratios=[1.0, 1.9], hspace=0.08)

    ax_overview = fig.add_subplot(grid[0, 0])
    overview_stop = min(float(timestamps[0] + overview_seconds), float(timestamps[-1]))
    overview_mask = timestamps <= overview_stop
    ax_overview.plot(
        timestamps[overview_mask],
        raw_signal[overview_mask],
        color="#111827",
        lw=0.8,
        label="Ripple channel raw",
    )
    if sharp_wave_diff is not None:
        scaled_sw = sharp_wave_diff[overview_mask] / np.nanmax(np.abs(sharp_wave_diff[overview_mask]))
        scaled_sw = scaled_sw * np.nanstd(raw_signal[overview_mask])
        ax_overview.plot(
            timestamps[overview_mask],
            scaled_sw,
            color="#059669",
            lw=1.0,
            alpha=0.85,
            label="Sharp-wave difference (scaled)",
        )

    overview_events = events.loc[events["start"] < overview_stop, ["start", "stop"]].to_numpy()
    for start, stop in overview_events:
        ax_overview.axvspan(start, stop, color="#f59e0b", alpha=0.18, lw=0)

    ax_overview.set_title("Joint SWR intervals on the ripple and sharp-wave channels")
    ax_overview.set_xlabel("Time (s)")
    ax_overview.set_ylabel("LFP")
    ax_overview.legend(loc="upper right", frameon=False)

    example_events = (
        events.sort_values(["peakNormedPower", "sharp_wave_peakNormedPower"], ascending=False)
        .head(n_examples)
        .sort_values("peaks")
    )
    example_grid = grid[1, 0].subgridspec(2, 3, hspace=0.30, wspace=0.22)
    axes = example_grid.subplots().ravel()
    half_window = int(round(example_window * fs))

    for panel_index, (ax, (_, event)) in enumerate(zip(axes, example_events.iterrows())):
        peak_idx = int(np.argmin(np.abs(timestamps - event["peaks"])))
        start_idx = max(0, peak_idx - half_window)
        stop_idx = min(len(timestamps), peak_idx + half_window + 1)
        local_time = (timestamps[start_idx:stop_idx] - event["peaks"]) * 1000.0

        ax.plot(local_time, raw_signal[start_idx:stop_idx], color="#111827", lw=1.1, label="Raw ripple")
        ax.plot(local_time, ripple_filtered[start_idx:stop_idx], color="#dc2626", lw=1.1, alpha=0.9, label="Ripple band")
        if sharp_wave_diff is not None:
            ax.plot(local_time, sharp_wave_diff[start_idx:stop_idx], color="#059669", lw=1.1, alpha=0.85, label="Sharp-wave diff")

        ax.axvspan(
            (event["start"] - event["peaks"]) * 1000.0,
            (event["stop"] - event["peaks"]) * 1000.0,
            color="#f59e0b",
            alpha=0.16,
            lw=0,
        )
        ax.axvline(0.0, color="#2563eb", ls="--", lw=1.0)
        ax.set_title(
            f"Ripple z={event['peakNormedPower']:.1f} | SW z={event['sharp_wave_peakNormedPower']:.1f}"
        )
        ax.set_xlabel("Time from ripple trough (ms)")
        ax.set_ylabel("LFP / filtered")

        feat_ax = ax.twinx()
        feat_ax.plot(local_time, ripple_power[start_idx:stop_idx], color="#991b1b", ls="--", lw=1.0, alpha=0.9, label="Ripple feature")
        feat_ax.axhline(params["high_threshold"], color="#991b1b", ls="--", lw=0.8, alpha=0.7)
        if sharp_wave_power is not None:
            feat_ax.plot(local_time, sharp_wave_power[start_idx:stop_idx], color="#047857", ls=":", lw=1.2, alpha=0.95, label="Sharp-wave feature")
            feat_ax.axhline(params["sharp_wave_high_threshold"], color="#047857", ls=":", lw=0.8, alpha=0.7)
        feat_ax.set_ylabel("Feature z-score")
        feat_ax.tick_params(axis="y", colors="#475569")

        if panel_index == 0:
            handles1, labels1 = ax.get_legend_handles_labels()
            handles2, labels2 = feat_ax.get_legend_handles_labels()
            ax.legend(handles1 + handles2, labels1 + labels2, loc="upper left", frameon=False, fontsize=8)

    for ax in axes[len(example_events):]:
        ax.axis("off")

    return fig


fig = plot_joint_swr_summary(
    ripples,
    timestamps,
    raw,
    ripple_filtered,
    ripple_power,
    fs,
    params=swr_params,
    sharp_wave_diff=sharp_wave_diff,
    sharp_wave_power=sharp_wave_power,
)
fig.suptitle("Joint sharp wave ripple detection summary", y=1.02, fontsize=15);


## 5) Save the final joint detections to a CellExplorer event file

Once the thresholds look good, rerun the detector with `save_mat=True`. Using the default `event_name="ripples"` writes a standard CellExplorer event file named `basename.ripples.events.mat`.

```python
detect_sharp_wave_ripples(
    basepath=basepath,
    low_threshold=swr_params["low_threshold"],
    high_threshold=swr_params["high_threshold"],
    sharp_wave_low_threshold=swr_params["sharp_wave_low_threshold"],
    sharp_wave_high_threshold=swr_params["sharp_wave_high_threshold"],
    min_duration=swr_params["min_duration"],
    max_duration=swr_params["max_duration"],
    sharp_wave_min_duration=swr_params["sharp_wave_min_duration"],
    sharp_wave_max_duration=swr_params["sharp_wave_max_duration"],
    noise_threshold=swr_params["noise_threshold"],
    merge_gap=swr_params["merge_gap"],
    boundary_mode=swr_params["boundary_mode"],
    save_mat=True,
    overwrite=True,
)
```

That file can then be loaded by CellExplorer or by `neuro_py.io.loading.load_ripples_events`.
